# Roger Federer, Rafael Nadal, Novak Djokovic 

## Who is the Greatest of All Time? 

###### A Data Science Project by Vishal Srikanth

### I. Intorduction

Importing Libraries

In [2]:
# Core data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    r2_score,
    mean_absolute_error,
 )
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans

# Gradient boosting
from xgboost import XGBClassifier, XGBRegressor

# Statistical/helper libraries
import scipy.stats as stats


# Date, time, and utilities
from datetime import datetime, date, timedelta
import time
import os
import re
import warnings
from pathlib import Path

# Notebook display helpers
from IPython.display import display

# Optional: cleaner notebook output
warnings.filterwarnings("ignore")

Data Ingestion

In [3]:
# Data ingestion: load all 19 CSV files from Raw Data using pd.read_csv
raw_data_dir = Path("Raw Data")

csv_filenames = [
    "charting-m-matches.csv",
    "charting-m-points-2010s.csv",
    "charting-m-points-2020s.csv",
    "charting-m-points-to-2009.csv",
    "charting-m-stats-KeyPointsReturn.csv",
    "charting-m-stats-KeyPointsServe.csv",
    "charting-m-stats-NetPoints.csv",
    "charting-m-stats-Overview.csv",
    "charting-m-stats-Rally.csv",
    "charting-m-stats-ReturnDepth.csv",
    "charting-m-stats-ReturnOutcomes.csv",
    "charting-m-stats-ServeBasics.csv",
    "charting-m-stats-ServeDirection.csv",
    "charting-m-stats-ServeInfluence.csv",
    "charting-m-stats-ShotDirection.csv",
    "charting-m-stats-ShotDirOutcomes.csv",
    "charting-m-stats-SnV.csv",
    "charting-m-stats-SvBreakSplit.csv",
    "charting-m-stats-SvBreakTotal.csv",
]

raw_dataframes = {}
for filename in csv_filenames:
    file_path = raw_data_dir / filename
    if not file_path.exists():
        raise FileNotFoundError(f"Missing expected file: {file_path}")
    dataset_key = Path(filename).stem.replace("-", "_")
    raw_dataframes[dataset_key] = pd.read_csv(file_path)

assert len(raw_dataframes) == 19, f"Expected 19 datasets, loaded {len(raw_dataframes)}"

print(f"Loaded {len(raw_dataframes)} CSV files successfully.")
display(
    pd.DataFrame(
        [
            {"dataset": k, "rows": v.shape[0], "columns": v.shape[1]}
            for k, v in raw_dataframes.items()
        ]
    ).sort_values("dataset").reset_index(drop=True)
 )

# Example usage:
# raw_dataframes['charting_m_matches'].head()

Loaded 19 CSV files successfully.


,dataset,rows,columns
0,charting_m_matches,7566,15
1,charting_m_points_2010s,357229,14
2,charting_m_points_2020s,547478,14
3,charting_m_points_to_2009,379569,14
4,charting_m_stats_KeyPointsReturn,60464,8
5,charting_m_stats_KeyPointsServe,60464,12
6,charting_m_stats_NetPoints,59396,11
7,charting_m_stats_Overview,56850,20
8,charting_m_stats_Rally,96706,13
9,charting_m_stats_ReturnDepth,270002,12


In [4]:
# Descriptive statistics for each loaded CSV
for dataset_name, df in raw_dataframes.items():
    print("=" * 90)
    print(f"Dataset: {dataset_name} | Shape: {df.shape}")
    print("=" * 90)

    # include='all' summarizes numeric + categorical columns
    summary = df.describe(include="all").transpose()
    display(summary)

Dataset: charting_m_matches | Shape: (7566, 15)


,count,unique,top,freq
match_id,7566,7565,20240915-M-Davis_Cup_World_Group-RR-Botic_Van_...,2
Player 1,7566,732,Roger Federer,489
Player 2,7566,759,Roger Federer,234
Pl 1 hand,7533,5,R,6712
Pl 2 hand,7533,6,R,6658
Date,7566,4439,20230117,13
Tournament,7566,430,Australian Open,591
Round,7565,15,R32,1279
Time,4094,446,3pm,351
Court,5609,369,Centre,1658


Dataset: charting_m_points_2010s | Shape: (357229, 14)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,357229,2232,20100622-M-Wimbledon-R128-Nicolas_Mahut-John_I...,980,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pt,357229.0,NaN,NaN,NaN,92.961882,71.186245,1.0,40.0,81.0,129.0,980.0
Set1,357229.0,NaN,NaN,NaN,0.485039,0.626555,0.0,0.0,0.0,1.0,2.0
Set2,357229.0,NaN,NaN,NaN,0.456525,0.61437,0.0,0.0,0.0,1.0,2.0
Gm1,357229.0,NaN,NaN,NaN,2.512442,2.49166,0.0,1.0,2.0,4.0,68.0
Gm2,357228.0,NaN,NaN,NaN,2.381941,2.511253,0.0,1.0,2.0,4.0,69.0
Pts,357229,114,0-0,57383,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gm#,357229.0,NaN,NaN,NaN,15.364735,11.588984,1.0,7.0,13.0,21.0,183.0
TbSet,357229,2,True,351298,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Svr,357229.0,NaN,NaN,NaN,1.494534,0.499971,1.0,1.0,1.0,2.0,2.0


Dataset: charting_m_points_2020s | Shape: (547478, 14)


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,51300,6327,19990704-M-Wimbledon-F-Pete_Sampras-Andre_Agassi,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
player,51300,779,Roger Federer,3794,NaN,NaN,NaN,NaN,NaN,NaN,NaN
row,51300,6,SnV,9553,NaN,NaN,NaN,NaN,NaN,NaN,NaN
snv_pts,51300.0,NaN,NaN,NaN,31.810175,32.906133,1.0,4.0,23.0,49.0,436.0
pts_won,51300.0,NaN,NaN,NaN,21.192047,22.45682,0.0,3.0,14.0,33.0,345.0
aces,51300.0,NaN,NaN,NaN,2.882417,5.176351,0.0,0.0,0.0,4.0,115.0
unret,51300.0,NaN,NaN,NaN,0.768616,2.31746,0.0,0.0,0.0,0.0,37.0
return_forced,51300.0,NaN,NaN,NaN,4.809357,6.831794,0.0,0.0,1.0,8.0,121.0
net_winner,51300.0,NaN,NaN,NaN,2.821598,3.667944,0.0,0.0,2.0,4.0,47.0
induced_forced,51300.0,NaN,NaN,NaN,1.562885,2.325256,0.0,0.0,1.0,2.0,35.0


Dataset: charting_m_stats_SvBreakSplit | Shape: (164871, 15)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,164871,7544,20180919-M-St_Petersburg-R16-Stan_Wawrinka-Kar...,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
player,164871,1002,Roger Federer,7894,NaN,NaN,NaN,NaN,NaN,NaN,NaN
row,164871,11,d,15114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_pts,164871.0,NaN,NaN,NaN,14.343335,11.791507,0.0,5.0,12.0,20.0,192.0
first_pts_won,164871.0,NaN,NaN,NaN,10.332181,8.822466,0.0,3.0,8.0,15.0,156.0
first_aces,164871.0,NaN,NaN,NaN,1.842513,2.630125,0.0,0.0,1.0,3.0,66.0
first_unret,164871.0,NaN,NaN,NaN,0.502399,1.317828,0.0,0.0,0.0,0.0,18.0
first_forced,164871.0,NaN,NaN,NaN,2.891267,3.232558,0.0,0.0,2.0,4.0,52.0
first_won_lte_3_shots,164871.0,NaN,NaN,NaN,6.469822,6.190736,0.0,2.0,5.0,9.0,129.0
second_pts,164871.0,NaN,NaN,NaN,8.794615,7.411823,0.0,3.0,7.0,12.0,82.0


Dataset: charting_m_stats_SvBreakTotal | Shape: (164871, 11)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_id,164871,7544,20180919-M-St_Petersburg-R16-Stan_Wawrinka-Kar...,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
player,164871,1002,Roger Federer,7894,NaN,NaN,NaN,NaN,NaN,NaN,NaN
row,164871,11,d,15114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pts,164871.0,NaN,NaN,NaN,23.13795,17.212717,1.0,10.0,19.0,32.0,260.0
pts_won,164871.0,NaN,NaN,NaN,14.822055,11.520902,0.0,6.0,12.0,21.0,200.0
aces,164871.0,NaN,NaN,NaN,1.910918,2.72528,0.0,0.0,1.0,3.0,69.0
unret,164871.0,NaN,NaN,NaN,0.566146,1.49245,0.0,0.0,0.0,0.0,21.0
forced_err,164871.0,NaN,NaN,NaN,3.263412,3.578364,0.0,0.0,2.0,5.0,72.0
pts_won_lte_3_shots,164871.0,NaN,NaN,NaN,8.235881,7.393456,0.0,3.0,6.0,12.0,161.0
first_in,164871.0,NaN,NaN,NaN,14.343335,11.791507,0.0,5.0,12.0,20.0,192.0


Data Merging

### Data Merging Report Structure
This section is split into small, readable units.
Each code cell defines one function, and the final cell runs the full pipeline.

#### 1) Constants
Define player and surface scopes used by every downstream step.

In [8]:
def get_project_constants():
    big_three = ["Roger Federer", "Rafael Nadal", "Novak Djokovic"]
    surfaces = ["Clay", "Grass", "Hard"]
    return big_three, surfaces

#### 2) Base Tables
Prepare the canonical match table and combined point table, then limit matches to the Big Three universe.

In [9]:
def prepare_base_tables(raw_dataframes, big_three):
    matches = raw_dataframes["charting_m_matches"].copy()
    matches = matches.rename(
        columns={
            "Player 1": "player1",
            "Player 2": "player2",
            "Date": "match_date",
            "Surface": "surface",
        }
    )
    matches["match_date"] = pd.to_datetime(
        matches["match_date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    )

    point_keys = [k for k in raw_dataframes if k.lower().startswith("charting_m_points")]
    points_all = pd.concat([raw_dataframes[k].copy() for k in point_keys], ignore_index=True)
    points_all["Pt"] = pd.to_numeric(points_all["Pt"], errors="coerce")
    points_all["PtWinner"] = pd.to_numeric(points_all["PtWinner"], errors="coerce")

    is_big_match = matches["player1"].isin(big_three) | matches["player2"].isin(big_three)
    matches_big = matches.loc[is_big_match].copy()
    return matches, points_all, matches_big

#### 3) Match Results
Infer each match winner from the final point to support win/loss and H2H metrics.

In [10]:
def infer_match_results(matches_big, points_all):
    last_point = (
        points_all.dropna(subset=["Pt"])
        .sort_values(["match_id", "Pt"])
        .groupby("match_id", as_index=False)
        .tail(1)
    )
    match_results = matches_big.merge(
        last_point[["match_id", "PtWinner"]],
        on="match_id",
        how="left",
    )
    match_results["winner"] = np.where(
        match_results["PtWinner"] == 1,
        match_results["player1"],
        np.where(match_results["PtWinner"] == 2, match_results["player2"], np.nan),
    )
    return match_results

#### 4) Master Player-Match Table
Create the core one-row-per-match-per-player table from Overview totals and add match context columns.

In [11]:
def build_master_from_overview(raw_dataframes, matches, big_three):
    overview = raw_dataframes["charting_m_stats_Overview"].copy()
    if "set" in overview.columns:
        overview = overview[overview["set"].astype(str).str.lower() == "total"]

    master = overview.merge(
        matches[["match_id", "player1", "player2", "surface", "match_date", "Tournament", "Round"]],
        on="match_id",
        how="left",
    )
    master["opponent"] = np.where(
        master["player"] == master["player1"],
        master["player2"],
        np.where(master["player"] == master["player2"], master["player1"], np.nan),
    )
    master = master[master["player"].isin(big_three)].copy()
    return master

#### 5) Stats Enrichment
Merge additional stats tables onto the master table using match_id + player and Total-level rows.

In [12]:
def enrich_master_with_stats(master, raw_dataframes, big_three):
    stats_sources = {
        "serve_basics": "charting_m_stats_ServeBasics",
        "serve_direction": "charting_m_stats_ServeDirection",
        "serve_influence": "charting_m_stats_ServeInfluence",
        "return_outcomes": "charting_m_stats_ReturnOutcomes",
        "return_depth": "charting_m_stats_ReturnDepth",
        "shot_direction": "charting_m_stats_ShotDirection",
        "shot_dir_outcomes": "charting_m_stats_ShotDirOutcomes",
        "net_points": "charting_m_stats_NetPoints",
        "snv": "charting_m_stats_SnV",
        "sv_break_total": "charting_m_stats_SvBreakTotal",
        "sv_break_split": "charting_m_stats_SvBreakSplit",
        "key_serve": "charting_m_stats_KeyPointsServe",
        "key_return": "charting_m_stats_KeyPointsReturn",
    }

    out = master.copy()
    for prefix, key_name in stats_sources.items():
        st = raw_dataframes[key_name].copy()
        if "row" in st.columns:
            st = st[st["row"].astype(str).str.lower() == "total"]
        if "set" in st.columns:
            st = st[st["set"].astype(str).str.lower() == "total"]
        st = st[st["player"].isin(big_three)].copy()
        st = st.drop(columns=["row", "set"], errors="ignore")
        non_keys = [c for c in st.columns if c not in ["match_id", "player"]]
        st = st.rename(columns={c: f"{prefix}__{c}" for c in non_keys})
        out = out.merge(st, on=["match_id", "player"], how="left")
    return out

#### 6) Point-Level Totals
Compute authoritative points won and points played from the point-by-point tables, then attach to the master table.

In [13]:
def add_point_level_totals(master, points_all, matches, big_three):
    point_base = points_all.merge(
        matches[["match_id", "player1", "player2", "surface", "match_date"]],
        on="match_id",
        how="left",
    )
    point_base = point_base[
        point_base["player1"].isin(big_three) | point_base["player2"].isin(big_three)
    ].copy()

    point_base["point_winner_name"] = np.where(
        point_base["PtWinner"] == 1,
        point_base["player1"],
        np.where(point_base["PtWinner"] == 2, point_base["player2"], np.nan),
    )

    points_won = (
        point_base.dropna(subset=["point_winner_name"])
        .groupby(["match_id", "point_winner_name"], as_index=False)
        .size()
        .rename(columns={"point_winner_name": "player", "size": "points_won"})
    )

    points_played = pd.concat(
        [
            point_base[["match_id", "player1"]].rename(columns={"player1": "player"}),
            point_base[["match_id", "player2"]].rename(columns={"player2": "player"}),
        ],
        ignore_index=True,
    )
    points_played = (
        points_played.dropna(subset=["player"])
        .groupby(["match_id", "player"], as_index=False)
        .size()
        .rename(columns={"size": "points_played"})
    )

    out = master.merge(points_won, on=["match_id", "player"], how="left")
    out = out.merge(points_played, on=["match_id", "player"], how="left")
    out["points_won"] = out["points_won"].fillna(0)
    out["points_played"] = out["points_played"].fillna(0)
    return out

#### 7) Career Window
Get debut and last match dates for each player based on match participation.

In [14]:
def build_career_window(matches_big, big_three):
    career_window = (
        matches_big.melt(
            id_vars=["match_id", "match_date", "surface"],
            value_vars=["player1", "player2"],
            value_name="player",
        )
        .query("player in @big_three")
        .groupby("player", as_index=False)
        .agg(debut_match=("match_date", "min"), last_match=("match_date", "max"))
    )
    return career_window

#### 8) KPI Builder
Aggregate player-level KPIs and compute derived metrics including DR and SAI.

In [15]:
def build_kpi_table(master, career_window, surfaces):
    out = master.copy()
    if "winner" in out.columns:
        out["is_win"] = (out["player"] == out["winner"]).astype(int)
    else:
        out["is_win"] = 0

    kpi = out.groupby("player", as_index=False).agg(
        career_wins=("is_win", "sum"),
        matches_played=("match_id", "nunique"),
        total_points_played=("points_played", "sum"),
        total_points_won=("points_won", "sum"),
        total_aces=("aces", "sum"),
        total_first_serve_points_won=("first_won", "sum"),
        total_second_serve_points_won=("second_won", "sum"),
        total_first_serves_in=("first_in", "sum"),
        total_second_serves_in=("second_in", "sum"),
        total_return_points=("return_pts", "sum"),
        total_return_points_won=("return_pts_won", "sum"),
        bp_faced=("bk_pts", "sum"),
        bp_saved=("bp_saved", "sum"),
    )

    kpi["pct_points_won"] = kpi["total_points_won"] / kpi["total_points_played"].replace(0, np.nan)
    kpi["pct_first_serve_points_won"] = kpi["total_first_serve_points_won"] / kpi["total_first_serves_in"].replace(0, np.nan)
    kpi["pct_second_serve_points_won"] = kpi["total_second_serve_points_won"] / kpi["total_second_serves_in"].replace(0, np.nan)

    kpi["service_points_won"] = kpi["total_first_serve_points_won"] + kpi["total_second_serve_points_won"]
    kpi["service_points_played"] = kpi["total_first_serves_in"] + kpi["total_second_serves_in"]
    service_won_rate = kpi["service_points_won"] / kpi["service_points_played"].replace(0, np.nan)
    return_won_rate = kpi["total_return_points_won"] / kpi["total_return_points"].replace(0, np.nan)
    kpi["dominance_ratio"] = return_won_rate / np.maximum(1 - service_won_rate, 0.00001)

    kpi["clutch_index_partial"] = 0.5 * (kpi["bp_saved"] / kpi["bp_faced"].replace(0, np.nan))
    kpi["avg_serve_speed"] = np.nan
    kpi["avg_serve_speed_by_surface"] = np.nan
    kpi = kpi.merge(career_window, on="player", how="left")

    wins_by_surface = (
        out.groupby(["player", "surface"], as_index=False)["is_win"].sum()
        .pivot(index="player", columns="surface", values="is_win")
        .fillna(0)
        .add_prefix("wins_")
        .reset_index()
    )
    kpi = kpi.merge(wins_by_surface, on="player", how="left")

    win_surface = out.groupby(["player", "surface"], as_index=False).agg(
        wins=("is_win", "sum"),
        matches=("match_id", "nunique"),
    )
    win_surface["win_rate_surface"] = win_surface["wins"] / win_surface["matches"].replace(0, np.nan)
    overall_wr = out.groupby("player", as_index=False).agg(
        overall_wins=("is_win", "sum"),
        overall_matches=("match_id", "nunique"),
    )
    overall_wr["win_rate_overall"] = overall_wr["overall_wins"] / overall_wr["overall_matches"].replace(0, np.nan)

    sai_rows = []
    for p in kpi["player"].tolist():
        wr_o = overall_wr.loc[overall_wr["player"] == p, "win_rate_overall"]
        if wr_o.empty:
            continue
        wr_o = float(wr_o.iloc[0])
        diffs = []
        for s in surfaces:
            wr_s = win_surface.loc[
                (win_surface["player"] == p) & (win_surface["surface"] == s),
                "win_rate_surface",
            ]
            wr_s = float(wr_s.iloc[0]) if not wr_s.empty else wr_o
            diffs.append((wr_s - wr_o) ** 2)
        sai_rows.append({"player": p, "surface_adaptability_index": 1 - np.sqrt((1 / 3) * np.sum(diffs))})

    kpi = kpi.merge(pd.DataFrame(sai_rows), on="player", how="left")
    return kpi

#### 9) Style and Rally Tables
Compute shot placement mix, serve placement mix, and rally-length proxies overall and by surface.

In [16]:
def build_placement_and_rally_tables(master):
    def directional_mix(df, group_cols, cols, prefix):
        tmp = df[group_cols + cols].copy()
        agg = tmp.groupby(group_cols, as_index=False)[cols].sum()
        denom = agg[cols].sum(axis=1).replace(0, np.nan)
        for c in cols:
            agg[f"{prefix}{c.split('__')[-1]}_pct"] = agg[c] / denom
        return agg.drop(columns=cols)

    shot_cols = [
        "shot_direction__crosscourt",
        "shot_direction__down_middle",
        "shot_direction__down_the_line",
        "shot_direction__inside_out",
        "shot_direction__inside_in",
    ]
    serve_dir_cols = [
        "serve_direction__deuce_wide",
        "serve_direction__deuce_middle",
        "serve_direction__deuce_t",
        "serve_direction__ad_wide",
        "serve_direction__ad_middle",
        "serve_direction__ad_t",
    ]

    shot_overall = directional_mix(master, ["player"], shot_cols, "shot_mix_")
    shot_by_surface = directional_mix(master, ["player", "surface"], shot_cols, "shot_mix_")
    serve_place_overall = directional_mix(master, ["player"], serve_dir_cols, "serve_place_")
    serve_place_by_surface = directional_mix(master, ["player", "surface"], serve_dir_cols, "serve_place_")

    rally = master.copy()
    rally["rally_length_proxy"] = rally["return_outcomes__total_shots"] / rally["return_outcomes__pts"].replace(0, np.nan)
    rally_proxy_surface = rally.groupby(["player", "surface"], as_index=False).agg(
        mean_rally_length_proxy=("rally_length_proxy", "mean")
    )
    return shot_overall, shot_by_surface, serve_place_overall, serve_place_by_surface, rally_proxy_surface

#### 10) Head-to-Head Builder
Generate pairwise H2H tables for Total, Clay, Grass, and Hard with requested rate metrics.

In [17]:
def build_h2h_tables(master, match_results):
    pairs = [
        ("Roger Federer", "Rafael Nadal"),
        ("Roger Federer", "Novak Djokovic"),
        ("Rafael Nadal", "Novak Djokovic"),
    ]

    def summarize(surface_filter=None):
        rows = []
        for a, b in pairs:
            m = match_results[
                ((match_results["player1"] == a) & (match_results["player2"] == b))
                | ((match_results["player1"] == b) & (match_results["player2"] == a))
            ].copy()
            if surface_filter is not None:
                m = m[m["surface"] == surface_filter].copy()

            match_ids = m["match_id"].dropna().unique()
            mp = master[master["match_id"].isin(match_ids)]
            mp = mp[mp["player"].isin([a, b])]

            matches_played = len(match_ids)
            wins_a = int((m["winner"] == a).sum())
            wins_b = int((m["winner"] == b).sum())

            for p, wins in [(a, wins_a), (b, wins_b)]:
                sub = mp[mp["player"] == p]
                first_sr = sub["first_won"].sum() / sub["first_in"].sum() if sub["first_in"].sum() else np.nan
                point_wr = sub["points_won"].sum() / sub["points_played"].sum() if sub["points_played"].sum() else np.nan
                ace_rate = sub["aces"].sum() / sub["serve_pts"].sum() if sub["serve_pts"].sum() else np.nan
                rows.append(
                    {
                        "pair": f"{a} vs {b}",
                        "surface": surface_filter if surface_filter is not None else "Total",
                        "player": p,
                        "matches_played": matches_played,
                        "wins": wins,
                        "win_rate": wins / matches_played if matches_played else np.nan,
                        "first_serve_win_rate": first_sr,
                        "point_win_rate": point_wr,
                        "ace_rate": ace_rate,
                    }
                )
        return pd.DataFrame(rows)

    return {
        "Total": summarize(None),
        "Clay": summarize("Clay"),
        "Grass": summarize("Grass"),
        "Hard": summarize("Hard"),
    }

#### 11) Pipeline Runner
Execute all functions in sequence and return a dictionary of final tables for reporting.

In [18]:
def run_big_three_report(raw_dataframes):
    big_three, surfaces = get_project_constants()
    matches, points_all, matches_big = prepare_base_tables(raw_dataframes, big_three)
    match_results = infer_match_results(matches_big, points_all)

    master = build_master_from_overview(raw_dataframes, matches, big_three)
    master = enrich_master_with_stats(master, raw_dataframes, big_three)
    master = add_point_level_totals(master, points_all, matches, big_three)
    master = master.merge(match_results[["match_id", "winner"]], on="match_id", how="left")

    career_window = build_career_window(matches_big, big_three)
    kpi = build_kpi_table(master, career_window, surfaces)
    shot_overall, shot_by_surface, serve_place_overall, serve_place_by_surface, rally_proxy_surface = build_placement_and_rally_tables(master)
    h2h = build_h2h_tables(master, match_results)

    return {
        "master_player_match": master,
        "kpi": kpi,
        "h2h_total": h2h["Total"],
        "h2h_clay": h2h["Clay"],
        "h2h_grass": h2h["Grass"],
        "h2h_hard": h2h["Hard"],
        "shot_overall": shot_overall,
        "shot_by_surface": shot_by_surface,
        "serve_place_overall": serve_place_overall,
        "serve_place_by_surface": serve_place_by_surface,
        "rally_proxy_surface": rally_proxy_surface,
    }

#### 12) Execute and Display
Run the pipeline and display the main report tables in a readable order.

In [20]:
report_tables = run_big_three_report(raw_dataframes)
print("Report tables generated.")

Report tables generated.


#### 13) Master Table Preview
This preview shows the unified player-match fact table for the Big Three.

In [ ]:
display(report_tables["master_player_match"].head(10))

#### 14) KPI Summary
This table contains the player-level KPI rollups used for GOAT comparison.

In [21]:
display(report_tables["kpi"].sort_values("player").reset_index(drop=True))

,player,career_wins,matches_played,total_points_played,total_points_won,total_aces,total_first_serve_points_won,total_second_serve_points_won,total_first_serves_in,total_second_serves_in,...,dominance_ratio,clutch_index_partial,avg_serve_speed,avg_serve_speed_by_surface,debut_match,last_match,wins_Clay,wins_Grass,wins_Hard,surface_adaptability_index
0,Novak Djokovic,419,550,93291.0,49849.0,3271,21777,8691,29725,15823,...,1.222305,0.319974,NaN,NaN,2005-01-17,2026-03-11,90,44,285,0.963933
1,Rafael Nadal,303,424,70773.0,37326.0,1476,16860,6019,23893,10800,...,1.177851,0.319284,NaN,NaN,2003-05-15,2024-11-19,140,23,140,0.880151
2,Roger Federer,571,722,125338.0,67376.0,6080,28777,12688,37377,22483,...,1.278006,0.329648,NaN,NaN,1998-07-04,2021-07-05,91,99,381,0.941814


#### 15) Head-to-Head Comparison
Each table below isolates one surface filter for the three rivalry pairs.

In [ ]:
display(report_tables["h2h_total"])

In [ ]:
display(report_tables["h2h_clay"])

In [ ]:
display(report_tables["h2h_grass"])

In [ ]:
display(report_tables["h2h_hard"])

#### 16) Shot Placement
View directional shot tendencies overall and by surface.

In [32]:
display(report_tables["shot_overall"])

,player,shot_mix_crosscourt_pct,shot_mix_down_middle_pct,shot_mix_down_the_line_pct,shot_mix_inside_out_pct,shot_mix_inside_in_pct
0,Novak Djokovic,0.446765,0.243248,0.144001,0.148741,0.017246
1,Rafael Nadal,0.445921,0.218874,0.157590,0.140359,0.037257
2,Roger Federer,0.422179,0.256608,0.144078,0.144131,0.033004


In [ ]:
display(report_tables["shot_by_surface"])

#### 17) Serve Placement
View serve-direction mix overall and by surface.

In [ ]:
display(report_tables["serve_place_overall"])

In [ ]:
display(report_tables["serve_place_by_surface"])

#### 18) Rally Length Proxy
Show mean rally-length proxy by surface.

In [ ]:
display(report_tables["rally_proxy_surface"])

Data Science: ML 

## II. Metric Definitions and GOAT Scoring
This section implements DR, CI, SAI, tactical innovation components by era, and a preference-driven GOAT simulator with bootstrap intervals.
All reported rates retain explicit numerators and denominators for auditability.

### Dominance Ratio: Aggregate Counts First
Dominance Ratio compares return-point pressure with the opponent's service-point resistance. Pool service and return counts first, then compute the rates:
$$\text{DR} = \frac{\text{return points won} / \text{return points played}}{\max(1 - (\text{service points won} / \text{service points played}), 0.00001)}$$
The proposal interprets higher DR as greater overall match control. The epsilon protects division by zero, but a near-zero raw denominator can still make DR extreme and unstable, so inspect its denominator flag and test sensitivity before interpreting it.

In [22]:
def compute_dominance_ratio(master, epsilon=1e-5):
    agg = master.groupby("player", as_index=False).agg(
        service_points_won=("first_won", "sum"),
        service_points_played=("first_in", "sum"),
        second_points_won=("second_won", "sum"),
        second_points_played=("second_in", "sum"),
        return_points_won=("return_pts_won", "sum"),
        return_points_played=("return_pts", "sum"),
    )

    agg["service_points_won_total"] = agg["service_points_won"] + agg["second_points_won"]
    agg["service_points_total"] = agg["service_points_played"] + agg["second_points_played"]
    agg["service_points_won_rate"] = agg["service_points_won_total"] / agg["service_points_total"].replace(0, np.nan)
    agg["return_points_won_rate"] = agg["return_points_won"] / agg["return_points_played"].replace(0, np.nan)

    raw_den = 1 - agg["service_points_won_rate"]
    agg["dr_denominator_raw"] = raw_den
    agg["dr_denominator_near_zero"] = raw_den.abs() < 0.01
    agg["dominance_ratio"] = agg["return_points_won_rate"] / np.maximum(raw_den, epsilon)
    return agg

### CI: Full vs Partial
The Clutch Index (CI), called pressure resistance in the proposal, summarizes results in high-leverage situations: saving break points, winning tiebreaks, and winning deciding sets.
$$\text{CI} = 0.5\left(\frac{\text{BP saved}}{\text{BP faced}}\right) + 0.3\left(\frac{\text{tiebreaks won}}{\text{tiebreaks played}}\right) + 0.2\left(\frac{\text{deciding sets won}}{\text{deciding sets played}}\right)$$
A zero denominator makes that component undefined. Report the full CI only when all three components are available, alongside their sample sizes; label any renormalized available-component result as partial.

In [23]:
def compute_clutch_index_components(raw_dataframes, master, big_three):
    matches, points_all, matches_big = prepare_base_tables(raw_dataframes, big_three)
    match_results = infer_match_results(matches_big, points_all)

    last_point = (
        points_all.dropna(subset=["Pt"])
        .sort_values(["match_id", "Pt"])
        .groupby("match_id", as_index=False)
        .tail(1)
    )[["match_id", "Set1", "Set2", "PtWinner"]].copy()
    last_point["Set1"] = pd.to_numeric(last_point["Set1"], errors="coerce").fillna(0)
    last_point["Set2"] = pd.to_numeric(last_point["Set2"], errors="coerce").fillna(0)
    last_point["PtWinner"] = pd.to_numeric(last_point["PtWinner"], errors="coerce")

    deciding = matches_big[["match_id", "player1", "player2", "Best of"]].merge(last_point, on="match_id", how="left")
    deciding["Best of"] = pd.to_numeric(deciding["Best of"], errors="coerce")
    deciding["final_set1"] = deciding["Set1"] + (deciding["PtWinner"] == 1).astype(float)
    deciding["final_set2"] = deciding["Set2"] + (deciding["PtWinner"] == 2).astype(float)
    deciding["total_sets"] = deciding["final_set1"] + deciding["final_set2"]
    deciding["deciding_set_played"] = deciding["total_sets"] == deciding["Best of"]

    deciding = deciding.merge(match_results[["match_id", "winner"]], on="match_id", how="left")
    player_match = deciding.melt(
        id_vars=["match_id", "winner", "deciding_set_played"],
        value_vars=["player1", "player2"],
        value_name="player",
    )[["match_id", "player", "winner", "deciding_set_played"]]
    player_match = player_match[player_match["player"].isin(big_three)].copy()
    player_match["deciding_set_won"] = (
        player_match["deciding_set_played"] & (player_match["player"] == player_match["winner"])
    ).astype(float)

    deciding_comp = player_match.groupby("player", as_index=False).agg(
        deciding_sets_played=("deciding_set_played", "sum"),
        deciding_sets_won=("deciding_set_won", "sum"),
    )

    bp_comp = master.groupby("player", as_index=False).agg(
        bp_saved=("bp_saved", "sum"),
        bp_faced=("bk_pts", "sum"),
    )

    clutch = bp_comp.merge(deciding_comp, on="player", how="left")
    clutch["tiebreaks_played"] = np.nan
    clutch["tiebreaks_won"] = np.nan
    clutch["bp_component"] = clutch["bp_saved"] / clutch["bp_faced"].replace(0, np.nan)
    clutch["tb_component"] = clutch["tiebreaks_won"] / clutch["tiebreaks_played"]
    clutch["deciding_component"] = clutch["deciding_sets_won"] / clutch["deciding_sets_played"].replace(0, np.nan)

    clutch["clutch_index_full"] = np.where(
        clutch[["bp_component", "tb_component", "deciding_component"]].notna().all(axis=1),
        0.5 * clutch["bp_component"] + 0.3 * clutch["tb_component"] + 0.2 * clutch["deciding_component"],
        np.nan,
    )

    weights = {"bp_component": 0.5, "tb_component": 0.3, "deciding_component": 0.2}
    available_w = clutch[[c for c in weights]].notna().mul([weights[c] for c in weights], axis=1).sum(axis=1)
    weighted_sum = (
        clutch["bp_component"].fillna(0) * 0.5
        + clutch["tb_component"].fillna(0) * 0.3
        + clutch["deciding_component"].fillna(0) * 0.2
    )
    clutch["clutch_index_partial_renorm"] = weighted_sum / available_w.replace(0, np.nan)
    clutch["clutch_partial_label"] = "Partial index (renormalized available components)"
    return clutch

### SAI: Surface Consistency Relative to Overall
The proposal introduces SAI as a measure of surface adaptability. This analysis uses the refined definition specified for the project: how consistently a player's win rates track their overall win rate across Clay, Grass, and Hard.
$$\text{SAI} = 1 - \sqrt{\frac{1}{3}\sum_{s \in \{\text{Clay, Grass, Hard}\}} (\text{WinRate}_s - \text{WinRate}_{\text{overall}})^2}$$
Note: the proposal describes surface win-rate standard deviation; this relative-to-overall formula is a distinct, refined definition. A higher SAI means less deviation from the overall rate, not necessarily a higher win rate. Require evidence on all three surfaces and show overall and surface win rates beside the index.

In [24]:
def compute_sai_table(master, surfaces):
    tmp = master.copy()
    tmp["is_win"] = (tmp["player"] == tmp["winner"]).astype(int)

    overall = tmp.groupby("player", as_index=False).agg(
        wins_overall=("is_win", "sum"),
        matches_overall=("match_id", "nunique"),
    )
    overall["win_rate_overall"] = overall["wins_overall"] / overall["matches_overall"].replace(0, np.nan)

    surface = tmp.groupby(["player", "surface"], as_index=False).agg(
        wins_surface=("is_win", "sum"),
        matches_surface=("match_id", "nunique"),
    )
    surface["win_rate_surface"] = surface["wins_surface"] / surface["matches_surface"].replace(0, np.nan)

    wide = surface.pivot(index="player", columns="surface", values="win_rate_surface").reset_index()
    for s in surfaces:
        if s not in wide.columns:
            wide[s] = np.nan

    out = overall.merge(wide, on="player", how="left")
    out["has_all_surfaces"] = out[surfaces].notna().all(axis=1)

    sq = ((out[surfaces].sub(out["win_rate_overall"], axis=0)) ** 2).sum(axis=1) / 3
    out["surface_adaptability_index"] = np.where(
        out["has_all_surfaces"],
        1 - np.sqrt(sq),
        np.nan,
    )
    return out

### Tactical Innovation: Era-Based Components
The proposal frames tactical innovation as adaptation across 2010–2014, 2015–2018, and 2019–2022. For each input, calculate the mean absolute change between adjacent era values, normalize it using preregistered bounds, and clip the result to [0, 1].
$$\text{Innovation} = 100\left(0.3\Delta\text{NetFreq} + 0.3\Delta\text{ServeSpeedVar} + 0.2\Delta\text{DropShotFreq} + 0.2\Delta\text{RacquetSpecShift}\right)$$
This exploratory score measures tactical change, not proven improvement. Keep the proposal's weights; if serve-speed variation or equipment-shift data are unsupported, report available components only and do not publish the full composite.

In [25]:
def compute_tactical_innovation(master):
    eras = [
        ("2010-2014", 2010, 2014),
        ("2015-2018", 2015, 2018),
        ("2019-2022", 2019, 2022),
    ]
    bounds = {
        "net_freq": 0.30,
        "serve_speed_var": np.nan,
        "drop_shot_freq": np.nan,
        "racquet_spec_shift": np.nan,
    }

    tmp = master.copy()
    tmp["year"] = pd.to_datetime(tmp["match_date"], errors="coerce").dt.year
    tmp["net_freq"] = tmp["net_points__net_pts"] / tmp["points_played"].replace(0, np.nan)

    era_rows = []
    for label, y0, y1 in eras:
        sub = tmp[(tmp["year"] >= y0) & (tmp["year"] <= y1)]
        agg = sub.groupby("player", as_index=False).agg(net_freq=("net_freq", "mean"))
        agg["era"] = label
        era_rows.append(agg)
    era_net = pd.concat(era_rows, ignore_index=True)

    out_rows = []
    for player, grp in era_net.groupby("player"):
        g = grp.set_index("era").reindex([e[0] for e in eras])
        deltas = []
        vals = g["net_freq"].to_list()
        for i in range(len(vals) - 1):
            if pd.notna(vals[i]) and pd.notna(vals[i + 1]):
                deltas.append(abs(vals[i + 1] - vals[i]))
        avg_abs_change = np.mean(deltas) if deltas else np.nan
        net_component = np.clip(avg_abs_change / bounds["net_freq"], 0, 1) if pd.notna(avg_abs_change) else np.nan

        out_rows.append(
            {
                "player": player,
                "net_freq_2010_2014": vals[0],
                "net_freq_2015_2018": vals[1],
                "net_freq_2019_2022": vals[2],
                "delta_netfreq_norm": net_component,
                "delta_servespeedvar_norm": np.nan,
                "delta_dropshotfreq_norm": np.nan,
                "delta_racquetspec_norm": np.nan,
                "innovation_score_partial": 100 * (0.3 * net_component) if pd.notna(net_component) else np.nan,
                "innovation_full_available": False,
            }
        )
    return pd.DataFrame(out_rows)

### Preference-Driven GOAT Simulator
Normalize eligible metrics against a documented reference population, then score with nonnegative weights summing to 1.
Bootstrap score uncertainty under each preference setting using match-level resampling.

In [30]:
def run_goat_simulator(raw_dataframes, report_tables, n_boot=300, seed=7):
    rng = np.random.default_rng(seed)
    master_big3 = report_tables["master_player_match"].copy()
    big_three = sorted(master_big3["player"].dropna().unique().tolist())
    surfaces = ["Clay", "Grass", "Hard"]

    dr_big3 = compute_dominance_ratio(master_big3)
    clutch_big3 = compute_clutch_index_components(raw_dataframes, master_big3, big_three)
    sai_big3 = compute_sai_table(master_big3, surfaces)

    metrics_big3 = (
        dr_big3[["player", "dominance_ratio"]]
        .merge(clutch_big3[["player", "clutch_index_partial_renorm", "clutch_index_full"]], on="player", how="left")
        .merge(
            sai_big3[["player", "win_rate_overall", "surface_adaptability_index", "has_all_surfaces", "matches_overall"]],
            on="player",
            how="left",
        )
        .rename(columns={
            "clutch_index_partial_renorm": "clutch_metric",
            "win_rate_overall": "overall_performance",
            "surface_adaptability_index": "versatility",
        })
    )

    matches_all = raw_dataframes["charting_m_matches"].copy().rename(
        columns={"Player 1": "player1", "Player 2": "player2", "Date": "match_date", "Surface": "surface"}
    )
    matches_all["match_date"] = pd.to_datetime(matches_all["match_date"].astype(str), format="%Y%m%d", errors="coerce")
    points_keys = [k for k in raw_dataframes if k.lower().startswith("charting_m_points")]
    points_all = pd.concat([raw_dataframes[k].copy() for k in points_keys], ignore_index=True)
    points_all["Pt"] = pd.to_numeric(points_all["Pt"], errors="coerce")
    points_all["PtWinner"] = pd.to_numeric(points_all["PtWinner"], errors="coerce")

    last_point = (
        points_all.dropna(subset=["Pt"])
        .sort_values(["match_id", "Pt"])
        .groupby("match_id", as_index=False)
        .tail(1)
    )[["match_id", "PtWinner"]]
    match_results_all = matches_all.merge(last_point, on="match_id", how="left")
    match_results_all["winner"] = np.where(
        match_results_all["PtWinner"] == 1,
        match_results_all["player1"],
        np.where(match_results_all["PtWinner"] == 2, match_results_all["player2"], np.nan),
    )

    overview_all = raw_dataframes["charting_m_stats_Overview"].copy()
    if "set" in overview_all.columns:
        overview_all = overview_all[overview_all["set"].astype(str).str.lower() == "total"]
    master_all = overview_all.merge(
        matches_all[["match_id", "player1", "player2", "surface", "match_date"]],
        on="match_id",
        how="left",
    )
    master_all = master_all.merge(match_results_all[["match_id", "winner"]], on="match_id", how="left")

    all_players = sorted(master_all["player"].dropna().unique().tolist())
    dr_ref = compute_dominance_ratio(master_all)
    clutch_ref = compute_clutch_index_components(raw_dataframes, master_all, all_players)
    sai_ref = compute_sai_table(master_all, surfaces)

    reference = (
        dr_ref[["player", "dominance_ratio"]]
        .merge(clutch_ref[["player", "clutch_index_partial_renorm"]].rename(columns={"clutch_index_partial_renorm": "clutch_metric"}), on="player", how="left")
        .merge(sai_ref[["player", "win_rate_overall", "surface_adaptability_index", "matches_overall", "has_all_surfaces"]].rename(columns={"win_rate_overall": "overall_performance", "surface_adaptability_index": "versatility"}), on="player", how="left")
    )
    reference = reference[reference["matches_overall"].fillna(0) >= 30].copy()

    metric_cols = ["dominance_ratio", "clutch_metric", "overall_performance", "versatility"]
    ref_bounds = {}
    for m in metric_cols:
        vals = reference[m].dropna()
        ref_bounds[m] = (vals.min(), vals.max()) if len(vals) else (np.nan, np.nan)

    def normalize_metric(x, lo, hi):
        if pd.isna(lo) or pd.isna(hi) or hi == lo or pd.isna(x):
            return np.nan
        return float(np.clip((x - lo) / (hi - lo), 0, 1))

    norm_big3 = metrics_big3.copy()
    for m in metric_cols:
        lo, hi = ref_bounds[m]
        norm_big3[f"norm_{m}"] = norm_big3[m].apply(lambda x: normalize_metric(x, lo, hi))

    scenarios = {
        "Balanced": {"dominance_ratio": 0.25, "clutch_metric": 0.25, "overall_performance": 0.25, "versatility": 0.25},
        "Clutch Priority": {"dominance_ratio": 0.20, "clutch_metric": 0.45, "overall_performance": 0.20, "versatility": 0.15},
        "Dominance Priority": {"dominance_ratio": 0.45, "clutch_metric": 0.20, "overall_performance": 0.20, "versatility": 0.15},
    }

    score_rows = []
    for scenario, w in scenarios.items():
        ws = np.array([w["dominance_ratio"], w["clutch_metric"], w["overall_performance"], w["versatility"]], dtype=float)
        ws = ws / ws.sum()
        cols = ["norm_dominance_ratio", "norm_clutch_metric", "norm_overall_performance", "norm_versatility"]
        tmp = norm_big3[["player"] + cols].copy()
        tmp["score"] = np.nansum(tmp[cols].values * ws, axis=1)
        tmp["scenario"] = scenario
        score_rows.append(tmp[["scenario", "player", "score"]])
    score_table = pd.concat(score_rows, ignore_index=True).sort_values(["scenario", "score"], ascending=[True, False])

    boot_rows = []
    for scenario, w in scenarios.items():
        ws = np.array([w["dominance_ratio"], w["clutch_metric"], w["overall_performance"], w["versatility"]], dtype=float)
        ws = ws / ws.sum()

        player_samples = {p: master_big3[master_big3["player"] == p].copy() for p in big_three}
        score_dist = {p: [] for p in big_three}

        for _ in range(n_boot):
            metric_boot = []
            for p in big_three:
                sub = player_samples[p]
                if sub.empty:
                    metric_boot.append({"player": p, "dominance_ratio": np.nan, "clutch_metric": np.nan, "overall_performance": np.nan, "versatility": np.nan})
                    continue

                idx = rng.integers(0, len(sub), len(sub))
                b = sub.iloc[idx].copy()
                b["is_win"] = (b["player"] == b["winner"]).astype(int)

                service_won = b["first_won"].sum() + b["second_won"].sum()
                service_played = b["first_in"].sum() + b["second_in"].sum()
                service_rate = service_won / service_played if service_played else np.nan
                return_rate = b["return_pts_won"].sum() / b["return_pts"].sum() if b["return_pts"].sum() else np.nan
                dr_b = return_rate / max(1 - service_rate, 1e-5) if pd.notna(service_rate) and pd.notna(return_rate) else np.nan

                bp_comp_b = b["bp_saved"].sum() / b["bk_pts"].sum() if b["bk_pts"].sum() else np.nan
                deciding_b = np.nan
                clutch_partial_b = bp_comp_b
                perf_b = b["is_win"].mean() if len(b) else np.nan

                surf_wr = b.groupby("surface", as_index=False).agg(wins=("is_win", "sum"), matches=("match_id", "nunique"))
                surf_wr["wr"] = surf_wr["wins"] / surf_wr["matches"].replace(0, np.nan)
                wr_map = {r["surface"]: r["wr"] for _, r in surf_wr.iterrows()}
                if all(s in wr_map and pd.notna(wr_map[s]) for s in surfaces) and pd.notna(perf_b):
                    sq = np.mean([(wr_map[s] - perf_b) ** 2 for s in surfaces])
                    sai_b = 1 - np.sqrt(sq)
                else:
                    sai_b = np.nan

                metric_boot.append({
                    "player": p,
                    "dominance_ratio": dr_b,
                    "clutch_metric": clutch_partial_b,
                    "overall_performance": perf_b,
                    "versatility": sai_b,
                })

            mb = pd.DataFrame(metric_boot)
            for m in metric_cols:
                lo, hi = ref_bounds[m]
                mb[f"norm_{m}"] = mb[m].apply(lambda x: normalize_metric(x, lo, hi))
            mb["score"] = np.nansum(
                mb[["norm_dominance_ratio", "norm_clutch_metric", "norm_overall_performance", "norm_versatility"]].values * ws,
                axis=1,
            )
            for _, r in mb.iterrows():
                score_dist[r["player"]].append(r["score"])

        for p in big_three:
            arr = np.array(score_dist[p], dtype=float)
            boot_rows.append({
                "scenario": scenario,
                "player": p,
                "score_boot_median": np.nanmedian(arr),
                "score_boot_p2_5": np.nanpercentile(arr, 2.5),
                "score_boot_p97_5": np.nanpercentile(arr, 97.5),
            })

    boot_table = pd.DataFrame(boot_rows).sort_values(["scenario", "score_boot_median"], ascending=[True, False])
    return {
        "dr_table": dr_big3,
        "clutch_table": clutch_big3,
        "sai_table": sai_big3,
        "innovation_table": compute_tactical_innovation(master_big3),
        "reference_population": reference,
        "normalized_big3": norm_big3,
        "goat_scores": score_table,
        "goat_bootstrap": boot_table,
    }

### Execute Metric Definitions and Simulator
Run advanced metric computations and display the core result tables for interpretation and sensitivity checks.

In [31]:
advanced_tables = run_goat_simulator(raw_dataframes, report_tables, n_boot=300, seed=42)

print("Dominance Ratio (with denominator diagnostics)")
display(advanced_tables["dr_table"])

print("Clutch components (full vs partial)")
display(advanced_tables["clutch_table"])

print("SAI with overall/surface win-rate evidence")
display(advanced_tables["sai_table"])

print("Tactical innovation components by era")
display(advanced_tables["innovation_table"])

print("Normalized Big Three metrics for simulator")
display(advanced_tables["normalized_big3"])

print("GOAT score ranking by preference scenario")
display(advanced_tables["goat_scores"])

print("Bootstrap score intervals by scenario")
display(advanced_tables["goat_bootstrap"])

Dominance Ratio (with denominator diagnostics)


,player,service_points_won,service_points_played,second_points_won,second_points_played,return_points_won,return_points_played,service_points_won_total,service_points_total,service_points_won_rate,return_points_won_rate,dr_denominator_raw,dr_denominator_near_zero,dominance_ratio
0,Novak Djokovic,21777,29725,8691,15823,19128,47267,30468,45548,0.668921,0.404680,0.331079,False,1.222305
1,Rafael Nadal,16860,23893,6019,10800,14526,36216,22879,34693,0.659470,0.401093,0.340530,False,1.177851
2,Roger Federer,28777,37377,12688,22483,25244,64278,41465,59860,0.692700,0.392732,0.307300,False,1.278006


Clutch components (full vs partial)


,player,bp_saved,bp_faced,deciding_sets_played,deciding_sets_won,tiebreaks_played,tiebreaks_won,bp_component,tb_component,deciding_component,clutch_index_full,clutch_index_partial_renorm,clutch_partial_label
0,Novak Djokovic,1996,3119,142,99.0,NaN,NaN,0.639949,NaN,0.697183,NaN,0.656301,Partial index (renormalized available components)
1,Rafael Nadal,1659,2598,105,69.0,NaN,NaN,0.638568,NaN,0.657143,NaN,0.643875,Partial index (renormalized available components)
2,Roger Federer,2266,3437,195,122.0,NaN,NaN,0.659296,NaN,0.625641,NaN,0.649680,Partial index (renormalized available components)


SAI with overall/surface win-rate evidence


...

,player,dominance_ratio,clutch_metric,clutch_index_full,overall_performance,versatility,has_all_surfaces,matches_overall,norm_dominance_ratio,norm_clutch_metric,norm_overall_performance,norm_versatility
0,Novak Djokovic,1.222305,0.656301,NaN,0.761818,0.963933,True,550,0.884927,0.937409,0.952130,0.928050
1,Rafael Nadal,1.177851,0.643875,NaN,0.714623,0.880151,True,424,0.793090,0.883073,0.874332,0.732145
2,Roger Federer,1.278006,0.649680,NaN,0.790859,0.941814,True,722,1.000000,0.908457,1.000000,0.876330


GOAT score ranking by preference scenario


,scenario,player,score
2,Balanced,Roger Federer,0.946197
0,Balanced,Novak Djokovic,0.925629
1,Balanced,Rafael Nadal,0.820660
5,Clutch Priority,Roger Federer,0.940255
3,Clutch Priority,Novak Djokovic,0.928453
4,Clutch Priority,Rafael Nadal,0.840689
8,Dominance Priority,Roger Federer,0.963141
6,Dominance Priority,Novak Djokovic,0.915332
7,Dominance Priority,Rafael Nadal,0.818193


Bootstrap score intervals by scenario


,scenario,player,score_boot_median,score_boot_p2_5,score_boot_p97_5
2,Balanced,Roger Federer,0.736657,0.706575,0.763356
0,Balanced,Novak Djokovic,0.687609,0.647318,0.725786
1,Balanced,Rafael Nadal,0.639652,0.593575,0.686836
5,Clutch Priority,Roger Federer,0.827506,0.784775,0.850000
3,Clutch Priority,Novak Djokovic,0.762674,0.722090,0.806236
4,Clutch Priority,Rafael Nadal,0.726558,0.669709,0.782739
8,Dominance Priority,Roger Federer,0.836932,0.801201,0.855269
6,Dominance Priority,Novak Djokovic,0.770761,0.730552,0.805278
7,Dominance Priority,Rafael Nadal,0.710528,0.658041,0.764751
